# 🔴 LLM Red Teaming — Notebook 1: Adversarial NLP Attacks

This notebook demonstrates adversarial text attacks on NLP models using the `llm_red_teaming` toolkit.

**What we cover:**
- 7 attack types across 4 perturbation levels
- Evaluation on the SST-2 sentiment dataset using GPT-4o as the victim model
- Comparison of accuracy drop across all attacks

**All heavy lifting lives in the `attacks/`, `targets/`, and `evaluate/` modules — this notebook is intentionally code-light.**

---

| Level | Attack | Meaning Preserved |
|---|---|---|
| Character | TextBugger | ❌ |
| Character | DeepWordBug | ❌ |
| Word | TextFooler | ~✅ |
| Word | BERTAttack | ✅ |
| Sentence | CheckList | ❌ |
| Sentence | StressTest | ❌ |
| Semantic | SemanticAttack | ✅ |

## 0 · Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
load_dotenv('../.env')

# ── Attack imports ────────────────────────────────────────────────────────────
from attacks.character import TextBugger, DeepWordBug
from attacks.word      import TextFooler, BERTAttack
from attacks.sentence  import CheckList, StressTest
from attacks.semantic  import SemanticAttack

# ── Target & evaluate imports ─────────────────────────────────────────────────
from targets.azure_openai import AzureOpenAITarget
from evaluate.metrics     import accuracy_drop, adversarial_report

print('✅ All modules loaded')

## 1 · Load SST-2 Dataset

In [ ]:
# Update this path to your local SST-2 folder
SST2_PATH = os.getenv('SST2_PATH', 'C:/Users/minwuu01/Data/SST-2')

dev_df = pd.read_csv(os.path.join(SST2_PATH, 'dev.tsv'), sep='\t')
print(f'Dev set: {len(dev_df)} rows')
dev_df.head()

## 2 · Instantiate Attacks & Target

In [ ]:
# Model target
target = AzureOpenAITarget()
print(target)

# All attacks — seeded for reproducibility
attacks = {
    'TextBugger':    TextBugger(seed=42),
    'DeepWordBug':   DeepWordBug(seed=42),
    'TextFooler':    TextFooler(seed=42),
    'BERTAttack':    BERTAttack(),        # loads BERT & sentence-transformer
    'CheckList':     CheckList(seed=42),
    'StressTest':    StressTest(seed=42),
    'SemanticAttack': SemanticAttack(),
}
print(f'\n{len(attacks)} attacks ready')

## 3 · Quick Sanity Check (5 samples)

In [ ]:
sample = dev_df.sample(5, random_state=42)['sentence'].tolist()

for name, atk in attacks.items():
    print(f'\n── {name} ──')
    for sent in sample:
        print(f'  original : {sent}')
        print(f'  attacked : {atk.attack(sent)}')
        print()

## 4 · Full Evaluation Loop

> **Note:** This calls the Azure OpenAI API for every sample × every attack.  
> With `NUM_SAMPLES=100` and 7 attacks × 2 calls each, expect ~1400 API calls.  
> Reduce `NUM_SAMPLES` for a quick test run.

In [ ]:
import time

NUM_SAMPLES = 50   # increase to 100 for full evaluation
SLEEP = 1.2        # seconds between calls to avoid rate limiting

eval_df = dev_df.head(NUM_SAMPLES).reset_index(drop=True)
results_by_attack: dict[str, dict] = {}

for attack_name, atk in attacks.items():
    print(f'\n{"="*50}')
    print(f'  Running {attack_name} on {NUM_SAMPLES} samples...')
    print(f'{"="*50}')

    orig_correct = 0
    atk_correct  = 0

    for _, row in eval_df.iterrows():
        text  = row['sentence']
        label = 'positive' if row['label'] == 1 else 'negative'

        orig_pred = target.get_sentiment(text)
        if orig_pred == label:
            orig_correct += 1

        attacked_text = atk.attack(text)
        atk_pred = target.get_sentiment(attacked_text)
        if atk_pred == label:
            atk_correct += 1

        time.sleep(SLEEP)

    metrics = accuracy_drop(orig_correct, atk_correct, NUM_SAMPLES)
    results_by_attack[attack_name] = metrics
    print(f'  → acc_drop = {metrics["acc_drop"]:.2%}')

print('\n✅ Evaluation complete')

## 5 · Results Summary

In [ ]:
report = adversarial_report(results_by_attack)
report.style.format({'original_acc': '{:.2%}', 'attacked_acc': '{:.2%}', 'acc_drop': '{:.2%}'})\
            .background_gradient(subset=['acc_drop'], cmap='Reds')

## 6 · Visualisation — Accuracy Drop by Attack

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=report,
    x='attack', y='acc_drop',
    palette='Reds_r', ax=ax
)
ax.set_title('Accuracy Drop by Attack Type (SST-2 / GPT-4o)', fontsize=14, fontweight='bold')
ax.set_xlabel('Attack')
ax.set_ylabel('Accuracy Drop')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('../results/01_accuracy_drop.png', dpi=150)
plt.show()

## 7 · Save Results

In [ ]:
os.makedirs('../results', exist_ok=True)
report.to_csv('../results/01_adversarial_results.csv', index=False)
print('Results saved to ../results/01_adversarial_results.csv')

## 8 · Observations

| Attack | Level | Expected Impact | Reason |
|---|---|---|---|
| **TextBugger** | Character | Low–Medium | Single char substitution — GPT-4o is robust to minor typos |
| **DeepWordBug** | Character | Low–Medium | Insert/swap/delete — may confuse tokenizer on rare words |
| **TextFooler** | Word | Medium | Synonym swap can shift sentiment polarity |
| **BERTAttack** | Word | Low–Medium | Context-preserving — by design preserves semantics |
| **CheckList** | Sentence | Low | Trailing noise — large models typically ignore it |
| **StressTest** | Sentence | Low | Tautology append — logically irrelevant |
| **SemanticAttack** | Semantic | Low–Medium | Meaning-preserving paraphrase — model may still classify correctly |

> **Key insight:** Large instruction-tuned models (GPT-4o) are considerably more robust than smaller fine-tuned classifiers to all of the above attacks. The highest accuracy drops are typically seen on character-level attacks applied to rare or domain-specific vocabulary.